# 02 Operations: Practice Problems

15 exercises on concepts from `01_intro` and `02_operations`.

In [ ]:
# Setup 1/2: make JAVA_HOME visible to the kernel before Spark starts
import os

os.environ["JAVA_HOME"] = "/Users/jerry/.sdkman/candidates/java/17.0.13-tem"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:{os.environ["PATH"]}'

In [ ]:
# Setup 2/2: start a local SparkSession
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("operations_practice")
    .master("local[*]")
    .getOrCreate()
)
spark.conf.set("spark.sql.shuffle.partitions", "5")

### Problem 1: Does the format change the schema?

Read `data/flight-data/csv/2015-summary.csv` (header on, schema inference on) and `data/flight-data/json/2015-summary.json` into two DataFrames. Print both schemas and note any difference in the inferred type of `count`.

In [ ]:
flight_df_2015 = (
    spark.read
    .format("csv")
    .option("inferSchema", "true") # infer data types of columns; needed for CSVs because otherwise they all become strings
    .option("header", "true") # use first line as column names
    .load("data/flight-data/csv/2015-summary.csv")
)

flight_json_df = (
    spark.read
    .format("json") # don't need inferSchema=true because JSON already contains some type information
    .load("data/flight-data/json/2015-summary.json")
)

flight_df_2015.printSchema()
flight_json_df.printSchema()

### Problem 2: Skip the inference pass

Define an explicit schema for the flight data (`DEST_COUNTRY_NAME` and `ORIGIN_COUNTRY_NAME` as strings, `count` as a 64-bit integer) and use it to load `data/flight-data/csv/2015-summary.csv` with header on and inference off. Confirm with `printSchema()` that `count` is a `long`.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

# Here although schema claimed nullable=False, when reading CSV the engine relaxes this constraint because if it cannot parse to a type, it will return null for column value
flight_schema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), False),
    StructField("ORIGIN_COUNTRY_NAME", StringType(), False),
    StructField("count", LongType(), False)
])

flight_df_2015 = (
    spark.read
    .format("csv")
    .schema(flight_schema)
    .option("header", "true")
    .load("data/flight-data/csv/2015-summary.csv")
)

flight_df_2015.printSchema()

### Problem 3: Foreign-origin flights into the US

From the 2015 flight CSV, return every route where `DEST_COUNTRY_NAME` is `"United States"` and `ORIGIN_COUNTRY_NAME` is not `"United States"`. Output only `ORIGIN_COUNTRY_NAME` and `count`, using `col(...)` for the references.

In [ ]:
from pyspark.sql.functions import col

flight_df_2015.where(
    "DEST_COUNTRY_NAME = 'United States'"
).where(
    "ORIGIN_COUNTRY_NAME != 'United States'"
).select(
    col("ORIGIN_COUNTRY_NAME"),
    col("count")
).show()

### Problem 4: Revenue per line item

Using `data/retail-data/by-day/2010-12-01.csv` (header + inferSchema), produce `InvoiceNo`, `Description`, and a new column `Revenue` equal to `Quantity * UnitPrice`. Do it twice: once with `selectExpr(...)`, and once with `select(...)` passing a `col()` arithmetic expression aliased as `Revenue`.

In [ ]:
from pyspark.sql.functions import expr, col, round

rel_retail_fp = "data/retail-data/by-day/2010-12-01.csv"
retail_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(rel_retail_fp)
)

retail_df.printSchema()

retail_df.select(
    col("InvoiceNo"),
    col("Description"),
    round(col("Quantity") * col("UnitPrice"), 2).alias("Revenue") # alternatively expr("ROUND(Quantity * UnitPrice, 2) AS Revenue")
).show()

retail_df.selectExpr(
    "InvoiceNo",
    "Description",
    "ROUND(Quantity * UnitPrice, 2) AS Revenue"
).show()

### Problem 5: Stamp the source and the revenue

On `data/retail-data/by-day/2010-12-01.csv`, use `withColumn` to add a constant column `SourceDate` holding the literal value `"2010-12-01"`, and a `Revenue` column (`Quantity * UnitPrice`). Show 5 rows.

In [ ]:
from pyspark.sql.functions import lit, expr

retail_df.show(3)

# Add constant column "SourceDate"
retail_df = retail_df.withColumn(
    "SourceDate",
    lit("2010-12-01")
)

retail_df.show(3)

# Add "Revenue" column
retail_df = retail_df.withColumn(
    "Revenue",
    expr("ROUND(Quantity * UnitPrice, 2)")
)

retail_df.show(3)

### Problem 6: Top 10 busiest routes

On the 2015 flight CSV, return the 10 routes with the highest `count`, most to least. Call `.explain()` on the sorted query and find the `Exchange` (shuffle) in the physical plan. In a comment, explain why `sort` is a wide transformation while the filter in Problem 3 was narrow.

In [ ]:
# Sort is a wide transformation because it requires shuffles across partitions as elements are moved around
# Also a sort needs to compare elements across multiple partitions it's not an identical operation that can be done in "parallel" on each partition
flight_df_2015.orderBy("count").limit(10).explain()

### Problem 7: Hand-built correction records

Build three `Row` objects: `("United States", "Canada", 9999)`, `("Egypt", "United States", 24)`, and `("United States", "Narnia", 1)`. Create a DataFrame from them with an explicit schema (strings + long), then `show()` the result and confirm the schema.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql import Row

FlightRow = Row("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count")

flight_schema = StructType([
    StructField("DEST_COUNTRY_NAME", StringType(), False),
    StructField("ORIIGN_COUNTRY_NAME", StringType(), False),
    StructField("count", LongType(), False)
])

new_flight_df = spark.createDataFrame(
    data=[
        FlightRow("United States", "Canada", 9999),
        FlightRow("Egypt", "United States", 24),
        FlightRow("United States", "Narnia", 1)
    ],
    schema=flight_schema
)

new_flight_df.printSchema()
new_flight_df.show()

### Problem 8: Six years of flights at once

Load every year in `data/flight-data/csv/` in a single read (point the reader at the folder, not one file). Compute the total `count` per `DEST_COUNTRY_NAME` across all years, then return the top 15 destinations by total, descending.

In [ ]:
flight_df = (
    spark.read
    .format("csv")
    .schema(flight_schema)
    .option("header", "true")
    .load("data/flight-data/csv/")
)

dest_cts = flight_df.groupBy("DEST_COUNTRY_NAME").count()
dest_cts.orderBy("count", ascending=False).limit(15).show()

### Problem 9: Where inference falls short

Read `data/bike-data/201508_trip_data.csv` (header + inferSchema) and print the schema. In a comment, identify at least two columns whose inferred type is not what you'd want for analysis (look at `Start Date` and `End Date`). Then select `col("Trip ID")` and `Duration` for the first 5 rows, noting how you reference a column whose name contains a space.

In [ ]:
trip_fp = "data/bike-data/201508_trip_data.csv"
trip_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(trip_fp)
)

trip_df.printSchema()
# start date, end date

trip_df.select(
    col("Trip ID"),
    col("Duration")
).limit(5).show()

In [ ]:
from pyspark.sql.functions import to_date

trip_df = trip_df.withColumn("Start Date", to_date("Start Date")).withColumn("End Date", to_date("End Date"))
trip_df.printSchema()

### Problem 10: Long rides by casual riders

On `data/bike-data/201508_trip_data.csv`, keep trips where `Subscriber Type` is `"Customer"` (not `"Subscriber"`) and `Duration` (seconds) is greater than `1800`. Return `Start Station`, `End Station`, and `Duration`, sorted by `Duration` descending, and show the top 10.

In [ ]:
bike_fp = "data/bike-data/201508_trip_data.csv"

bike_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(bike_fp)
)

bike_df.where(
    col("Subscriber Type") == "Customer"
).where(
    col("Duration") > 1800
).select(
    col("Start Station"),
    col("End Station"),
    col("Duration")
).orderBy("Duration", ascending=False).show(10)

### Problem 11: Flag the bulk orders

On `data/retail-data/by-day/2010-12-01.csv`, using only `selectExpr`, return `InvoiceNo`, a `TotalValue` column (`Quantity * UnitPrice`), and a boolean `IsBulk` that is `true` when `Quantity >= 50`.

In [ ]:
retail_fp = "data/retail-data/by-day/2010-12-01.csv"
retail_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(retail_fp)
)

retail_df.printSchema()

retail_df.selectExpr(
    "InvoiceNo",
    "ROUND(Quantity * UnitPrice, 2) AS TotalValue",
    "Quantity >= 50 AS IsBulk"
).show()

### Problem 12: Two ways to name a column

On the 2015 flight CSV, select the `count` column twice in one `select`: once via `col("count")` and once via `df["count"]` (alias the second as `count_bracket`). Add a third expression that renames `count` to `num_flights`. In a comment, give one situation where `df["count"]` can't be used but `col("count")` can.

In [ ]:
flight_df.printSchema()
flight_df.show(3)

flight_df.select(
    col("count"),
    flight_df["count"].alias("count_bracket"),
    col("count").alias("num_flights")
).show(5)

In [ ]:
spark.stop()